In [1]:
! gdown 1v4UNEdB0UVKdKfGmO1SPP1q3YBk7CVcL

Downloading...
From: https://drive.google.com/uc?id=1v4UNEdB0UVKdKfGmO1SPP1q3YBk7CVcL
To: /content/Вариант47_Б24-215_Новиков.xlsx
100% 2.13M/2.13M [00:00<00:00, 112MB/s]


In [2]:
import numpy as np
import pandas as pd
from scipy import stats


In [3]:
np.random.seed(100)
seed = 100
df = pd.read_excel('/content/Вариант47_Б24-215_Новиков.xlsx')
df.rename({'Насолько Вам важна цена пива': 'Насколько Вам важна цена пива'}, axis=1, inplace=True)
df.rename(columns={'Насколько дешевым должно быть идеальное пиво?': 'Насколько дешевым должно быть Идеальное пиво?'}, inplace=True)
N = len(df)

In [4]:
df.columns

Index(['Unnamed: 0', 'Код Респондента', 'Возраст', 'Возрастная група', 'Пол',
       'Станция', 'Насколько дешевым должно быть Идеальное пиво?',
       'Насколько крепким должно быть Идеальное пиво (2.5% - 15%)?',
       'Насколько высокий показатель IBU/EBU (горечь)  должно иметь Идеальное пиво?',
       'Насколько плотным должно быть Идеальное пиво?',
       'Насколько необычный вкус должен быть у Идеального пива?',
       'Насколько Вам важна цена пива', 'Насколько Вам важна Крепость пива?',
       'Насколько Вам важна горечь пива?',
       'Насколько Вам важна плотность пива?',
       'Насколько Вам важна необчность вкуса пива?',
       'Насколько Вы считаете, "Охота крепкое" дешевое пиво?',
       'Насколько Вы считаете, "Охота крепкое" крепкое пиво?',
       'Насколько Вы считаете, "Охота крепкое" имеет высокий показатель IBU/EBU (горечь) ?',
       'Насколько Вы считаете, "Охота крепкое" плотное пиво?',
       'Насколько Вы считаете, "Охота крепкое" имеет необычный вкус?',
     

## Расчет отклонения от идеальной точки

In [5]:
columns = df.columns.to_list()

def find_column(word, column_list):
  res = []
  for i in column_list:
    if word in i:
      res.append(i)
  return res

ideal_sort_beer = find_column('Идеальное пиво', columns)

important_cols = find_column('Вам важна', columns)

sort_beer = {
    "Охота крепкое": find_column("Охота крепкое", columns),
    "Guinness": find_column("Guinness", columns),
    "Bud": find_column("Bud", columns),
    "IPA от Волковской пивоварни": find_column("IPA от Волковской пивоварни", columns)
}

def deviation(row, ideal_cols, important_cols, sort_bear):
  dev = 0
  for ideal_col, important_col, brand_col in zip(ideal_cols, important_cols, sort_bear):
    ideal = row[ideal_col]
    importance = row[important_col]
    brand = row[brand_col]
    dev += importance * abs(ideal - brand)
  return dev

for brand, brand_cols in sort_beer.items():
    df[f'Dev_{brand}'] = df.apply(
        lambda row: deviation(row, ideal_sort_beer, important_cols, brand_cols),
        axis=1
    )

best_distance = float('inf')

for brand in sort_beer.keys():
    mean_distance = df[f'Dev_{brand}'].mean()
    print(f'{brand}, {mean_distance:.2f}')
    if mean_distance < best_distance:
        best_distance = mean_distance
        best_brand = brand

print(f'Лучший бренд: {best_brand}')

best_brand_column = f'Dev_{best_brand}'

Охота крепкое, 35.37
Guinness, 35.73
Bud, 40.99
IPA от Волковской пивоварни, 34.13
Лучший бренд: IPA от Волковской пивоварни


## Оценка дисперсии с помощью разведочной совокупности

In [6]:
n4 = int(N * 0.04)
def sample_method(n, replace, seed=100):
    sample = df.sample(n=n, replace=replace, random_state=seed)

    variance_a = sample[best_brand_column].var()

    return sample, variance_a

pilot_sample_a, variance_a = sample_method(n4, replace=True)

print(f"n = {len(pilot_sample_a)}, дисперсия = {variance_a:.2f}")


n = 600, дисперсия = 241.60


In [7]:
def sample_method_mech(n, replace=False, start_i=0):
    data_sorted = df.sort_values('Код Респондента').reset_index(drop=True)
    N = len(data_sorted)
    step = int(N / n)
    start = (N // 2 + start_i) % N

    indices = [(start + i * step) % N for i in range(n)]
    sample = data_sorted.iloc[indices]
    variance = sample[best_brand_column].var()
    return sample, variance

pilot_sample_mech, variance_mech = sample_method_mech(n=n4)
variance_mech = pilot_sample_mech[best_brand_column].var()

print(f"Механический: n = {len(pilot_sample_mech)}, дисперсия = {variance_mech:.2f}")

Механический: n = 600, дисперсия = 232.21


In [8]:
df['group_typical'] = df['Пол'] + '_' + df['Возрастная група']

def sample_method_5b(n, replace, seed=100):
    group_samples = []
    group_variances = []
    group_sizes = []

    groups = df['group_typical'].unique()

    for group_name in groups:
        group_data = df[df['group_typical'] == group_name]
        group_size = len(group_data)

        group_proportion = group_size / len(df)
        sample_size = max(1, int(np.ceil(n * group_proportion)))


        group_sample = group_data.sample(n=sample_size, replace=replace, random_state=seed)

        group_variance = group_sample[best_brand_column].var()

        group_samples.append(group_sample)
        group_variances.append(group_variance)
        group_sizes.append(sample_size)

    pilot_sample_b = pd.concat(group_samples, ignore_index=True)
    numerator = sum(n * v for n, v in zip(group_sizes, group_variances))
    denominator = sum(group_sizes)
    variance = numerator / denominator

    return pilot_sample_b, variance

pilot_sample_b, variance_b = sample_method_5b(n=n4,replace=True)
print(f"Типический: n = {len(pilot_sample_b)}, дисперсия = {variance_b:.2f}")

Типический: n = 604, дисперсия = 210.64


In [9]:
def sample_method_5c(n, replace, seed=100):
    stations = df['Станция'].unique()

    avg_per_station = len(df) / len(stations)
    num_stations_needed = int(np.ceil(n / avg_per_station))

    np.random.seed(seed)

    selected_stations = np.random.choice(stations, size=num_stations_needed, replace=replace)

    pilot_sample_c = df[df['Станция'].isin(selected_stations)]

    station_stats = pilot_sample_c.groupby('Станция')[best_brand_column].agg(['mean', 'count'])

    overall_mean = pilot_sample_c[best_brand_column].mean()

    numerator = sum(
        row['count'] * (row['mean'] - overall_mean)**2
        for i, row in station_stats.iterrows()
    )
    denominator = station_stats['count'].sum()
    variance_c = numerator / denominator

    return pilot_sample_c, variance_c

pilot_sample_c, variance_c = sample_method_5c(n=n4, replace=False)
print(f"Серийный: n = {len(pilot_sample_c)}, дисперсия = {variance_c:.2f}")

Серийный: n = 614, дисперсия = 2.43


## Оценка устойчивости результата по разведочной совокупности

In [18]:
def run_100_iterations(sample_function, n, replace):
    wins = {"Охота крепкое": 0, "Guinness": 0, "Bud": 0, "IPA от Волковской пивоварни": 0}
    for iteration in range(100):
        if sample_function is sample_method_mech:
            sample, _ = sample_function(n, replace=replace, start_i=iteration)
        else:
            sample, _ = sample_function(n, replace=replace, seed=100+iteration)

        brand_means = {}
        for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]:
            col = f'Dev_{brand}'
            brand_means[brand] = sample[col].mean()

        winner = min(brand_means, key=brand_means.get)
        wins[winner] += 1

    return wins

methods = {
    'Случайный': sample_method,
    'Механический': sample_method_mech,
    'Типический': sample_method_5b,
    'Серийный': sample_method_5c
}

results = {}

for method_name, sample_func in methods.items():
    wins = run_100_iterations(sample_func, n=n4, replace=True)
    results[method_name] = wins


print("СВОДНАЯ ТАБЛИЦА УСТОЙЧИВОСТИ")

print(f"\n{'Метод':<25} | {"Охота крепкое":<8} | {"Guinness":<8} | {"Bud":<8} | {"IPA от Волковской пивоварни":<8}")
print("-" * 95)

for method_name, wins in results.items():
    print(f"{method_name:<25} | {wins["Охота крепкое"]:<13} | {wins["Guinness"]:<8} | {wins["Bud"]:<8} | {wins["IPA от Волковской пивоварни"]:<8}")

СВОДНАЯ ТАБЛИЦА УСТОЙЧИВОСТИ

Метод                     | Охота крепкое | Guinness | Bud      | IPA от Волковской пивоварни
-----------------------------------------------------------------------------------------------
Случайный                 | 3             | 1        | 0        | 96      
Механический              | 4             | 0        | 0        | 96      
Типический                | 8             | 3        | 0        | 89      
Серийный                  | 3             | 0        | 0        | 97      


## Расчет предельной ошибки среднего

In [26]:
alpha = 0.05

n_a = len(pilot_sample_a)
n_mech = len(pilot_sample_mech)
n_b = len(pilot_sample_b)
n_c = len(pilot_sample_c)

def calculate_margin_of_error(variance, n, alpha, fpc=None):
    if fpc is None:
        fpc = np.sqrt((len(df)-n)/(len(df)-1))
    t_value = stats.t.ppf(1 - alpha/2, df=n-1)
    std_error = np.sqrt(variance / n)
    margin_of_error = t_value * std_error * fpc
    return margin_of_error, t_value, std_error

delta_a, t_a, se_a = calculate_margin_of_error(variance_a, n_a, alpha)
delta_mech, t_mech, se_mech = calculate_margin_of_error(variance_mech, n_mech, alpha, fpc=1)
delta_b, t_b, se_b = calculate_margin_of_error(variance_b, n_b, alpha)

M = df["Станция"].nunique()
avg_per_station = len(df) / len(df["Станция"].unique())
num_stations_needed = int(np.ceil(n_c / avg_per_station))
fpc_c = np.sqrt(1-num_stations_needed/M)
delta_c, t_c, se_c = calculate_margin_of_error(variance_c, num_stations_needed, alpha, fpc=fpc_c)

print(f"\n ИНТЕРПРЕТАЦИЯ:")
print("-" * 75)
print(f"    5a: Истинное среднее находится в интервале ±{delta_a:.2f} от выборочного")
print(f"    5a_mech: Истинное среднее находится в интервале ±{delta_mech:.2f} от выборочного")
print(f"    5b: Истинное среднее находится в интервале ±{delta_b:.2f} от выборочного")
print(f"    5c: Истинное среднее находится в интервале ±{delta_c:.2f} от выборочного")
print("-" * 75)

methods_errors = {
    '5a (Случайный)': delta_a,
    '5a_mech (Механический)': delta_mech,
    '5b (Типический)': delta_b,
    '5c (Серийный)': delta_c
}
best_method = min(methods_errors, key=methods_errors.get)

print(f"\n Наименьшая предельная ошибка у метода: {best_method}")



 ИНТЕРПРЕТАЦИЯ:
---------------------------------------------------------------------------
    5a: Истинное среднее находится в интервале ±1.22 от выборочного
    5a_mech: Истинное среднее находится в интервале ±1.22 от выборочного
    5b: Истинное среднее находится в интервале ±1.14 от выборочного
    5c: Истинное среднее находится в интервале ±0.97 от выборочного
---------------------------------------------------------------------------

 Наименьшая предельная ошибка у метода: 5c (Серийный)


## Целевая ошибка

In [27]:
delta_a_new = delta_a / 2
delta_mech_new = delta_mech / 2
delta_b_new = delta_b / 2
delta_c_new = delta_c / 2

## Расчет объема выборки

In [28]:
alpha = 0.05
t_value = stats.t.ppf(1 - alpha/2, df=N-1)

def sample_size_without_replacement(variance, delta, N, t):
    return int(np.ceil(N * (t**2) * variance) / (N * (delta**2) + (t**2) * variance))


def sample_size_with_replacement(variance, delta, t):
    return int(np.ceil((t**2 * variance) / (delta**2)))

n_9a_wh = sample_size_without_replacement(variance_a, delta_a_new, N, t_value)
n_9a_w = sample_size_with_replacement(variance_a, delta_a_new, t_value)
n_9mech_wh = sample_size_without_replacement(variance_mech, delta_mech_new, N, t_value)
n_9c_wh = sample_size_without_replacement(variance_c, delta_c_new, N, t_value)
n_9c_w = sample_size_with_replacement(variance_c, delta_c_new, t_value)
n_9b_wh = sample_size_without_replacement(variance_b, delta_b_new, N, t_value)
n_9b_w = sample_size_with_replacement(variance_b, delta_b_new, t_value)

print(f"\n{'Метод':<30} | {'Тип':<12} | {'Дисперсия':<10} | {'n_min':<10}")
print("-" * 75)
print(f"{'Случайный бесповторный':<30} | {'Бесповт.':<12} | {variance_a:<10.2f} | {n_9a_wh:<10}")
print(f"{'Случайный повторный':<30} | {'Повторный':<12} | {variance_a:<10.2f} | {n_9a_w:<10}")
print(f"{'Механический бесповторный':<30} | {'Бесповт.':<12} | {variance_mech:<10.2f} | {n_9mech_wh:<10}")
print(f"{'Серийный бесповторный':<30} | {'Бесповт.':<12} | {variance_c:<10.2f} | {n_9c_wh:<10}")
print(f"{'Серийный повторный':<30} | {'Повторный':<12} | {variance_c:<10.2f} | {n_9c_w:<10}")
print(f"{'Типический бесповторный':<30} | {'Бесповт.':<12} | {variance_b:<10.2f} | {n_9b_wh:<10}")
print(f"{'Типический повторный':<30} | {'Повторный':<12} | {variance_b:<10.2f} | {n_9b_w:<10}")


Метод                          | Тип          | Дисперсия  | n_min     
---------------------------------------------------------------------------
Случайный бесповторный         | Бесповт.     | 241.60     | 2135      
Случайный повторный            | Повторный    | 241.60     | 2491      
Механический бесповторный      | Бесповт.     | 232.21     | 2062      
Серийный бесповторный          | Бесповт.     | 2.43       | 39        
Серийный повторный             | Повторный    | 2.43       | 40        
Типический бесповторный        | Бесповт.     | 210.64     | 2148      
Типический повторный           | Повторный    | 210.64     | 2508      


In [29]:
#10
alpha = 0.05

def sample_size_without_replacement_iterative(variance, delta, N, alpha):

    z_value = stats.norm.ppf(1 - alpha/2)
    n_0 = int(np.ceil((N * (z_value**2) * variance) / (N * (delta**2) + (z_value**2) * variance)))

    t_value = stats.t.ppf(1 - alpha/2, df=n_0-1)
    n_1 = int(np.ceil((N * (t_value**2) * variance) / (N * (delta**2) + (t_value**2) * variance)))

    return n_0, n_1, z_value, t_value

def sample_size_with_replacement_iterative(variance, delta, alpha):

    z_value = stats.norm.ppf(1 - alpha/2)
    n_0 = int(np.ceil((z_value**2 * variance) / (delta**2)))

    t_value = stats.t.ppf(1 - alpha/2, df=n_0-1)
    n_1 = int(np.ceil((t_value**2 * variance) / (delta**2)))

    return n_0, n_1, z_value, t_value

n0_9a_wh, n1_9a_wh, z_9a_wh, t_9a_wh = sample_size_without_replacement_iterative(variance_a, delta_a_new, N, alpha)
n0_9a_w, n1_9a_w, z_9a_w, t_9a_w = sample_size_with_replacement_iterative(variance_a, delta_a_new, alpha)
n0_9mech_wh, n1_9mech_wh, z_9mech_wh, t_9mech_wh = sample_size_without_replacement_iterative(variance_mech, delta_mech_new, N, alpha)
n0_9c_wh, n1_9c_wh, z_9c_wh, t_9c_wh = sample_size_without_replacement_iterative(variance_c, delta_c_new, N, alpha)
n0_9c_w, n1_9c_w, z_9c_w, t_9c_w = sample_size_with_replacement_iterative(variance_c, delta_c_new, alpha)
n0_9b_wh, n1_9b_wh, z_9b_wh, t_9b_wh = sample_size_without_replacement_iterative(variance_b, delta_b_new, N, alpha)
n0_9b_w, n1_9b_w, z_9b_w, t_9b_w = sample_size_with_replacement_iterative(variance_b, delta_b_new, alpha)

print(f"\n{'Метод':<30} | {'n*_0 (z)':<10} | {'n*_1 (t)':<10} | {'Разница':<10}")
print("-" * 60)
print(f"{'Случайный бесповторный':<30} | {n0_9a_wh:<10} | {n1_9a_wh:<10} | {n1_9a_wh-n0_9a_wh:<10}")
print(f"{'Случайный повторный':<30} | {n0_9a_w:<10} | {n1_9a_w:<10} | {n1_9a_w-n0_9a_w:<10}")
print(f"{'Механический бесповторный':<30} | {n0_9mech_wh:<10} | {n1_9mech_wh:<10} | {n1_9mech_wh-n0_9mech_wh:<10}")
print(f"{'Серийный бесповторный':<30} | {n0_9c_wh:<10} | {n1_9c_wh:<10} | {n1_9c_wh-n0_9c_wh:<10}")
print(f"{'Серийный повторный':<30} | {n0_9c_w:<10} | {n1_9c_w:<10} | {n1_9c_w-n0_9c_w:<10}")
print(f"{'Типический бесповторный':<30} | {n0_9b_wh:<10} | {n1_9b_wh:<10} | {n1_9b_wh-n0_9b_wh:<10}")
print(f"{'Типический повторный':<30} | {n0_9b_w:<10} | {n1_9b_w:<10} | {n1_9b_w-n0_9b_w:<10}")


Метод                          | n*_0 (z)   | n*_1 (t)   | Разница   
------------------------------------------------------------
Случайный бесповторный         | 2136       | 2138       | 2         
Случайный повторный            | 2490       | 2493       | 3         
Механический бесповторный      | 2062       | 2064       | 2         
Серийный бесповторный          | 40         | 43         | 3         
Серийный повторный             | 40         | 43         | 3         
Типический бесповторный        | 2149       | 2151       | 2         
Типический повторный           | 2508       | 2510       | 2         


In [31]:
# 11

def sample_method_11a(n, replace, seed):
    sample = df.sample(n=n, replace=replace, random_state=seed)

    variance_a = sample[best_brand_column].var()

    return sample, variance_a

def sample_method_mech_11(n, seed):
    data_sorted = df.sort_values('Код Респондента').reset_index(drop=True)
    N = len(data_sorted)
    step = int(N / n)
    start = (N // 2 + seed) % N

    indices = [(start + i * step) % N for i in range(n)]
    sample = data_sorted.iloc[indices]
    variance = sample[best_brand_column].var()

    return sample, variance

def sample_method_11b(n, replace, seed):
    group_samples = []
    group_variances = []
    group_sizes = []

    groups = df['group_typical'].unique()

    for group_name in groups:
        group_data = df[df['group_typical'] == group_name]
        group_size = len(group_data)

        group_proportion = group_size / len(df)
        sample_size = max(1, int(np.ceil(n * group_proportion)))


        group_sample = group_data.sample(n=sample_size, replace=replace, random_state=seed)

        group_variance = group_sample[best_brand_column].var()


        group_samples.append(group_sample)
        group_variances.append(group_variance)
        group_sizes.append(sample_size)

    pilot_sample_b = pd.concat(group_samples, ignore_index=True)
    numerator = sum(n * v for n, v in zip(group_sizes, group_variances))
    denominator = sum(group_sizes)
    variance = numerator / denominator

    return pilot_sample_b, variance

def sample_method_11c(n, replace, seed):
    stations = df['Станция'].unique()

    avg_per_station = len(df) / len(stations)
    num_stations_needed = int(np.ceil(n / avg_per_station))
    np.random.seed(seed)

    selected_stations = np.random.choice(stations, size=num_stations_needed, replace=replace)

    pilot_sample_c = df[df['Станция'].isin(selected_stations)]

    station_stats = pilot_sample_c.groupby('Станция')[best_brand_column].agg(['mean', 'count'])

    overall_mean = pilot_sample_c[best_brand_column].mean()

    numerator = sum(
        row['count'] * (row['mean'] - overall_mean)**2
        for i, row in station_stats.iterrows()
    )
    denominator = station_stats['count'].sum()

    variance_c = numerator / denominator

    return pilot_sample_c, variance_c


sample_11a_wh, _ = sample_method_11a(n=n1_9a_wh, replace=False, seed=100)
sample_11a_w, _ = sample_method_11a(n=n1_9a_w, replace=True, seed=100)
sample_11mech_wh, _ = sample_method_mech_11(n=n1_9mech_wh, seed=100)
sample_11c_wh, _ = sample_method_11c(n=n1_9c_wh, replace=False, seed=100)
sample_11c_w, _ = sample_method_11c(n=n1_9c_w, replace=True, seed=100)
sample_11b_wh, _ = sample_method_11b(n=n1_9b_wh, replace=False, seed=100)
sample_11b_w, _ = sample_method_11b(n=n1_9b_wh, replace=True, seed=100)

In [33]:
# 12

methods_12 = {
    '12a_wh': lambda seed: sample_method_11a(n=n1_9a_wh, replace=False, seed=seed),
    '12a_w': lambda seed: sample_method_11a(n=n1_9a_w, replace=True, seed=seed),
    '12mech_wh': lambda seed: sample_method_mech_11(n=n1_9mech_wh, seed=seed),
    '12c_wh': lambda seed: sample_method_11c(n=n1_9c_wh, replace=False, seed=seed),
    '12c_w': lambda seed: sample_method_11c(n=n1_9c_w, replace=True, seed=seed),
    '12b_wh': lambda seed: sample_method_11b(n=n1_9b_wh, replace=False, seed=seed),
    '12b_w': lambda seed: sample_method_11b(n=n1_9b_w, replace=True, seed=seed),
}

all_winners_12 = {}
all_brand_means_12 = {} # Для 14ого пункта

for method_name, sample_func in methods_12.items():
    winners = []
    brand_means_all = {brand: [] for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]} # Для 14ого пункта
    for seed in range(1000):
        sample, _ = sample_func(seed=100+seed)

        # Для 14ого пункта
        for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]:
            brand_means_all[brand].append(sample[f'Dev_{brand}'].mean())


        brand_means = {brand: sample[f'Dev_{brand}'].mean()
            for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]
        }
        winner = min(brand_means, key=brand_means.get)
        winners.append(winner)

    all_winners_12[method_name] = winners
    all_brand_means_12[method_name] = brand_means_all

print("Пункт 12 выполнен")

Пункт 12 выполнен


In [42]:
# 13

wins_13 = {}

for method_name, winners in all_winners_12.items():
    wins = {"Охота крепкое": 0, "Guinness": 0, "Bud": 0, "IPA от Волковской пивоварни": 0}
    for winner in winners:
        wins[winner] += 1
    wins_13[method_name] = wins

print(f"\n{'Метод':<10} | {'Победитель':<27} | {'% Побед':<6} | {"Охота крепкое":<8} | {"Guinness":<8} | {"Bud":<8} | {"IPA от Волковской пивоварни":<8}")
print("-" * 118)

best_method = None
best_pct = 0

for method_name, wins in wins_13.items():
    best_brand = max(wins, key=wins.get)
    pct = wins[best_brand] / 10 # wins[best_brand] * 100 / 1000
    print(f"{method_name:<10} | {best_brand:<10} | {pct:<7.1f} | {wins["Охота крепкое"]:<13} | {wins["Guinness"]:<8} | {wins["Bud"]:<8} | {wins["IPA от Волковской пивоварни"]:<8}")

    if pct > best_pct:
        best_pct = pct
        best_method = method_name
        best_brand_overall = best_brand

print("=" * 117)

print(f"  Наиболее устойчивый метод: {best_method}")
print(f"  Бренд {best_brand_overall} побеждает в {best_pct:.1f}% случаев")


Метод      | Победитель                  | % Побед | Охота крепкое | Guinness | Bud      | IPA от Волковской пивоварни
----------------------------------------------------------------------------------------------------------------------
12a_wh     | IPA от Волковской пивоварни | 99.8    | 2             | 0        | 0        | 998     
12a_w      | IPA от Волковской пивоварни | 100.0   | 0             | 0        | 0        | 1000    
12mech_wh  | IPA от Волковской пивоварни | 99.6    | 4             | 0        | 0        | 996     
12c_wh     | IPA от Волковской пивоварни | 56.1    | 228           | 207      | 4        | 561     
12c_w      | IPA от Волковской пивоварни | 56.7    | 252           | 179      | 2        | 567     
12b_wh     | IPA от Волковской пивоварни | 100.0   | 0             | 0        | 0        | 1000    
12b_w      | IPA от Волковской пивоварни | 100.0   | 0             | 0        | 0        | 1000    
  Наиболее устойчивый метод: 12a_w
  Бренд IPA от Волковской 

In [43]:
# 14

alpha = 0.05
z_critical = stats.norm.ppf(1 - alpha/2)

true_means = {
    brand: df[f'Dev_{brand}'].mean()
    for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]
}

print(f"\n{'Метод':<15} | {'Бренд':<27} | {'Среднее':<10} | {'Std':<10} | {'Истинное':<10} | {'Ошибка':<10}")
print("-" * 95)

avg_error = []
avg_std = []
results_15 = {}


for method_name, brand_means_all in all_brand_means_12.items():
    errors = []
    stds = []
    method_results = {}
    for brand in ["Охота крепкое", "Guinness", "Bud", "IPA от Волковской пивоварни"]:
        avg_estimate = np.mean(brand_means_all[brand])
        std_estimate = np.std(brand_means_all[brand])
        true_value = true_means[brand]
        t_stat, p_value = stats.ttest_1samp(brand_means_all[brand], true_value)

        solution = 1 if p_value > 0.05 else 0
        method_results[brand] = {
            'p_value': p_value,
            'solution': solution
        }

        error = abs(avg_estimate - true_value)
        errors.append(error)
        stds.append(std_estimate)

        print(f"{method_name:<15} | {brand:<27} | {avg_estimate:<10.2f} | {std_estimate:<10.2f} | {true_value:<10.2f} | {error:<10.2f}")

    results_15[method_name] = method_results

    avg_error.append(np.mean(errors))
    avg_std.append(np.mean(stds))


Метод           | Бренд                       | Среднее    | Std        | Истинное   | Ошибка    
-----------------------------------------------------------------------------------------------
12a_wh          | Охота крепкое               | 35.36      | 0.37       | 35.37      | 0.01      
12a_wh          | Guinness                    | 35.72      | 0.35       | 35.73      | 0.01      
12a_wh          | Bud                         | 40.99      | 0.33       | 40.99      | 0.00      
12a_wh          | IPA от Волковской пивоварни | 34.12      | 0.31       | 34.13      | 0.01      
12a_w           | Охота крепкое               | 35.36      | 0.37       | 35.37      | 0.00      
12a_w           | Guinness                    | 35.73      | 0.35       | 35.73      | 0.01      
12a_w           | Bud                         | 40.99      | 0.34       | 40.99      | 0.00      
12a_w           | IPA от Волковской пивоварни | 34.12      | 0.31       | 34.13      | 0.00      
12mech_wh       | Охо

In [44]:
# 15 Вывод результатов

method_names = list(all_brand_means_12.keys())

for method_name, brand_results in results_15.items():
    passed = sum(1 for r in brand_results.values() if r['solution'] == 1)

    if passed == 4:
        status = "Все оценки несмещённые"
    else:
        status = "Есть смещённые оценки"

    print(f"\n{method_name}: {status} ({passed}/4 брендов)")


12a_wh: Все оценки несмещённые (4/4 брендов)

12a_w: Все оценки несмещённые (4/4 брендов)

12mech_wh: Все оценки несмещённые (4/4 брендов)

12c_wh: Есть смещённые оценки (3/4 брендов)

12c_w: Все оценки несмещённые (4/4 брендов)

12b_wh: Все оценки несмещённые (4/4 брендов)

12b_w: Все оценки несмещённые (4/4 брендов)
